In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# General & Fixed Parameters
w_dyke = 450              # Full width of the dyke (m)
a = w_dyke / 2            # Half-width of the dyke (m)
x_max = a + 1000          # Spatial boundary 1000m past contact (m)
Th_C = 20                 # Host rock ambient temperature (°C)
sec_per_year = 365.25 * 24 * 3600
dx = 5.0                  # Spatial resolution (m)
x = np.arange(0, x_max + dx, dx)
Nx = len(x)

# List of Scenarios
scenarios = [
    {
        'label': 'Scenario A: 1000 yrs, T=1000°C, κ=1.3e-6',
        'Ti_C': 1000,
        'kappa': 1.3e-6,
        'active_years': 1000,
        'color': '#1f77b4'  # blue
    },
    {
        'label': 'Scenario B: 1000 yrs, T=1000°C, κ=1.0e-6',
        'Ti_C': 1000,
        'kappa': 1.0e-6,
        'active_years': 1000,
        'color': '#2ca02c'  # green
    },
    {
        'label': 'Scenario C: 1000 yrs, T=1000°C, κ=0.7e-6',
        'Ti_C': 1000,
        'kappa': 0.7e-6,
        'active_years': 1000,
        'color': '#a21d3c'  # red
    }
]

# 3. Figure Layout
fig, ax = plt.subplots(figsize=(8.5, 6))

T_target = 580.0
distance_from_contact_km = (x - a) / 1000.0

# Simulation & Plotting Loop 
for sc in scenarios:
    Ti_C = sc['Ti_C']
    kappa = sc['kappa']
    active_years = sc['active_years']
    cooling_years = 20000
    total_years = active_years + cooling_years
    
    # Calculate stable time step per scenario
    dt = (dx**2) / (2.5 * kappa)
    total_steps = int((total_years * sec_per_year) / dt)
    active_steps = int((active_years * sec_per_year) / dt)
    
    # Initial conditions
    T = np.ones(Nx) * Th_C
    T[x <= a] = Ti_C
    max_temperatures_C = np.copy(T)
    
    # Finite-difference solver loop
    for step in range(total_steps):
        T_old = np.copy(T)
        T[1:-1] = T_old[1:-1] + kappa * dt / dx**2 * (T_old[2:] - 2*T_old[1:-1] + T_old[:-2])
        T[0] = T[1]
        T[-1] = Th_C
        
        if step < active_steps:
            T[x <= a] = Ti_C
            
        max_temperatures_C = np.maximum(max_temperatures_C, T)
        
    # Mask out profiles for plotting (0 to 1.0 km)
    mask_main = (distance_from_contact_km >= 0) & (distance_from_contact_km <= 1.0)
    plot_dist_main = distance_from_contact_km[mask_main]
    plot_temps_main = max_temperatures_C[mask_main]
    
    # Calculate 580°C intersection distance
    idx_after = np.where(plot_temps_main < T_target)[0][0]
    idx_before = idx_after - 1
    x1, x2 = plot_dist_main[idx_before], plot_dist_main[idx_after]
    y1, y2 = plot_temps_main[idx_before], plot_temps_main[idx_after]
    x_intersect = x1 + (T_target - y1) * (x2 - x1) / (y2 - y1)
    
    full_label = f"{sc['label']} ({x_intersect*1000:.1f} m reset)"
    
    # Plot profile on main axes
    ax.plot(plot_dist_main, plot_temps_main, color=sc['color'], linewidth=2.2, label=full_label)
    
    # Intersection marker on main plot
    ax.plot(x_intersect, T_target, marker='o', color=sc['color'], markersize=5, zorder=5)

# 580°C Reference Line 
ax.axhline(y=T_target, color='dimgrey', linestyle='--', linewidth=1.2, label='Magnetite Curie Temp (580°C)')

# Axes & Legends
ax.set_xlim(0, 1.0)
ax.set_ylim(0, 1300)
ax.set_xlabel('Distance from Contact (km)', fontsize=14, labelpad=10)
ax.set_ylabel('Peak Temperature (°C)', fontsize=14, labelpad=10)
ax.set_yticks([0, 200, 400, 600, 800, 1000, 1200])
ax.tick_params(axis='both', which='major', direction='out', length=6, width=1.2, labelsize=11)
ax.legend(loc='upper right', frameon=True, fontsize=12)

plt.subplots_adjust(left=0.12, right=0.95, top=0.95, bottom=0.12)
plt.show()